# Import library

In [1]:
import sklearn
import pandas as pd
import seaborn as sns
import numpy as np 
import re 
import matplotlib.pyplot as plt 
import os 
import warnings
import kagglehub
import shutil 

warnings.filterwarnings('ignore')

# Fetching Data

In [2]:
def fetch(): 
    if not (os.path.exists('Watches.csv')): 
        data_path = kagglehub.dataset_download('philmorekoung11/luxury-watch-listings')
        source = os.path.join (data_path, 'Watches.csv')
        destination = os.path.join (os.getcwd())
        shutil.copy2 (source, destination)
    data_path = os.path.join(os.getcwd(), 'Watches.csv')
    return data_path
data_path = fetch() 
watches = pd.read_csv(data_path)

In [3]:
watches

,Unnamed: 0,name,price,brand,model,ref,mvmt,casem,bracem,yop,cond,sex,size,condition
0,0,Audemars Piguet Royal Oak Offshore Chronograph...,"$43,500",Audemars Piguet,Royal Oak Offshore Chronograph,26237ST.OO.1000ST.01,NaN,NaN,NaN,2019,Unworn,Men's watch/Unisex,42 mm,NaN
1,1,Audemars Piguet Royal Oak Selfwinding\n39mm Bl...,"$71,500",Audemars Piguet,Royal Oak Selfwinding,15300ST.OO.1220ST.02,NaN,NaN,NaN,2012,Very good,Men's watch/Unisex,39 mm,NaN
2,2,Audemars Piguet Royal Oak Chronograph\nBlue Di...,"$79,191",Audemars Piguet,Royal Oak Chronograph,26331ST,Automatic,Steel,Steel,Unknown,Unworn,NaN,41 mm,NaN
3,3,Audemars Piguet Royal Oak Chronograph\nSelfwin...,"$108,000",Audemars Piguet,Royal Oak Chronograph,26715ST.OO.1356ST.01,Automatic,Steel,Steel,2022 (Approximation),New,Men's watch/Unisex,38 mm,NaN
4,4,Audemars Piguet Royal Oak Offshore Chronograph...,"$27,500",Audemars Piguet,Royal Oak Offshore Chronograph,26170ST.OO.1000ST.01,Automatic,Steel,Steel,Unknown,Very good,Men's watch/Unisex,42 x 54 mm,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284486,6438,Zenith Defy El Primero\n21 TITANIUM 95.9000.90...,"$9,790",Zenith,Defy El Primero,95.9000.9004/78.M9000,NaN,NaN,NaN,2022,Very good,Men's watch/Unisex,44 mm,NaN
284487,6439,Zenith Chronomaster Sport\n41mm White 03.3100....,"$8,450",Zenith,Chronomaster Sport,03.3100.3600/69.M3100,NaN,NaN,NaN,2021 (Approximation),Very good,Men's watch/Unisex,41 mm,NaN
284488,6440,Zenith El Primero\n50th Anniversary A386 Limited,"$16,500",Zenith,El Primero,30.A386.400/69.C807,NaN,NaN,NaN,2019,Very good,Men's watch/Unisex,38 mm,NaN
284489,6441,Zenith Chronomaster Sport\nWhite Dial Chronogr...,"$9,000",Zenith,Chronomaster Sport,03.3100.3600/69.M3100,NaN,NaN,NaN,2021,Unworn,Men's watch/Unisex,41 mm,NaN


In [4]:
summary_df = pd.DataFrame({
    'feature' : watches.columns, 
    '# instances': len(watches), 
    '# of nan/null values': watches.isna().sum().values, 
    '% of nan/null values': (watches.isna().sum() / len(watches) * 100).values, 
})

summary_df.sort_values('% of nan/null values', ascending = False)

,feature,# instances,# of nan/null values,% of nan/null values
13,condition,284491,212922,74.843141
6,mvmt,284491,196685,69.135755
8,bracem,284491,174896,61.476813
7,casem,284491,164271,57.742073
11,sex,284491,95805,33.675934
10,cond,284491,75987,26.709808
1,name,284491,72585,25.513988
5,ref,284491,43152,15.168142
4,model,284491,30466,10.708950
12,size,284491,23597,8.294463


#### Data Overview
* Most of the features are in 'string' datatype -> requires conversion to int/float
* All features have nan/null values, especially 'condition', 'mvmt', 'bracem', 'casem' with over 50% nan

#### Strategy: 
* Clean the target data first: price with 0.14% nan/null values
* Clean the other features from least to most nan values
* Drop 'Unnamed' feature
* Extract and drop features with over 50% nan/null values 

# Create a copy of dataset for cleaning

In [5]:
watches_cleaned = watches.copy()

# Clean the Target: Price

In [6]:
watches['price'].unique()

<StringArray>
[ '$43,500',  '$71,500',  '$79,191', '$108,000',  '$27,500',  '$41,000',
  '$37,800',  '$32,182',  '$36,473',  '$35,000',
 ...
   '$3,678',  '$14,812',  '$26,208',  '$27,165', '$113,316',  '$87,192',
  '$58,464',  '$18,245',  '$20,068',  '$32,108']
Length: 36831, dtype: str

Watches is written in the format $xxx,xxx
* Use regex to remove '$', ',' with ''
* Use pd.to_numeric to convert string to number

In [7]:
watches_cleaned['price'] = watches['price'].replace ({r'\$': '', ',' : ''}, regex = True)
watches_cleaned['price'] = pd.to_numeric(watches_cleaned['price'], errors='coerce')
watches_cleaned.dropna(subset='price', inplace = True)

# Clean 'brand' feature

In [8]:
watches['name'].unique()

<StringArray>
[                            'Audemars Piguet Royal Oak Offshore Chronograph\n'25th Anniversary'',
                                          'Audemars Piguet Royal Oak Selfwinding\n39mm Blue Dial',
             'Audemars Piguet Royal Oak Chronograph\nBlue Dial - BV Limited Edition 10 Dlc - Pvd',
                       'Audemars Piguet Royal Oak Chronograph\nSelfwinding Chronograph Blue Dial',
   'Audemars Piguet Royal Oak Offshore Chronograph\nCronografo 42mm Panda Dial 26170STOO1000ST01',
 'Audemars Piguet Royal Oak Offshore Diver\n2018 Unworn Royal Oak Offshore Diver (Limit 200pics)',
                     'Audemars Piguet Royal Oak Offshore Diver Chronograph\n26703ST.OO.A051CA.01',
                           'Audemars Piguet Royal Oak Offshore Chronograph\n26470IO.OO.A006CA.01',
                                      'Audemars Piguet Royal Oak Dual Time\n26120ST.OO.1220ST.03',
                         'Audemars Piguet Royal Oak Offshore Chronograph\n26401.RO.OO.A002.CA.0

Analysis: 
* The 'brand' is often written in the beginning of the 'name'

Strategy:
* Create a set storing the existing brands 
* Iterate through each name and look up in the set
* Return if one 'brand' appears in the 'name'
* To avoid overlapping, brands with longer name should appear first in the set

In [9]:
# List of brands
brand_list = watches_cleaned['brand'].dropna().unique().tolist() 
brand_set = set(brand_list)
brand_set = sorted ([str(b) for b in brand_set if pd.notnull(b)], key = len, reverse = True)
print (brand_set)

['Vacheron Constantin', 'Jaeger-LeCoultre', 'A. Lange & Söhne', 'Audemars Piguet', 'Patek Philippe', 'Meistersinger', 'Richard Mille', 'Montblanc', 'Breitling', 'TAG Heuer', 'Hamilton', 'Longines', 'Panerai', 'Cartier', 'Zenith', 'Hublot', 'Tissot', 'Rolex', 'NOMOS', 'Seiko', 'Omega', 'Tudor', 'Sinn', 'Ebel', 'Oris', 'Rado', 'Edox', 'IWC']


In [10]:
def find_brand (name, original_brand): 
    if (str(original_brand) != 'nan'): return original_brand

    if pd.isnull (name) or pd.isna(name): 
        return original_brand
    
    for brand in brand_set: 
        if brand.lower() in name.lower():
            return brand 
        
    return original_brand

watches_cleaned['brand'] = watches_cleaned.apply (
    lambda row: find_brand(row['name'], row['brand']), 
    axis = 1, 
)
print ('Check nan:', watches_cleaned['brand'].isna().any())

print ('Check null:', watches_cleaned['brand'].isnull().any())

Check nan: False
Check null: False


# Clean 'yop' feature

Strategy:
* Create a pattern regex for year

* Look this regex up in all of the features

* Document the % of year patter showing up in different features

In [11]:
print (watches['yop'].value_counts()); print()
print ('Unknown proportion: ' , len(watches[watches['yop'] == 'Unknown']) / len (watches['yop']) * 100) 

year_list = watches['yop'].astype(str).str.extractall(r'(\d{4})')[0]
year_set = {int(y) for y in year_list if 1000 < int(y) < 2027}
min_yop = min(year_set)
max_yop = max(year_set)
print ('min_yop: ', min_yop)
print ('max_yop: ', max_yop)

print ('Nan proportion: ', watches['yop'].isna().mean() * 100)

yop
Unknown                         95957
2023                            49227
2022                            20654
2021                            10501
2023 (Approximation)             6136
                                ...  
1989, 1990, 1992, 1993, 1995        1
1995, 1996, 1997, 1998              1
1800                                1
2006, 2007, 2016, 2017, 2018        1
1650 (Approximation)                1
Name: count, Length: 410, dtype: int64

Unknown proportion:  33.72936226453561
min_yop:  1559
max_yop:  2024
Nan proportion:  0.04710166578204583


In [12]:
# Year Pattern 
year_pattern = r'(15[5-9]\d|1[6-9]\d{2}|200\d|201\d|202[0-4])'

# Check year patter proportion in all features
proportion = pd.DataFrame({
    '% yop': watches_cleaned.apply (lambda col: 
    col.dropna().astype (str).str.contains(year_pattern).mean() * 100), 
    'count yop': watches_cleaned.apply (lambda col: 
    col.dropna().astype (str).str.contains(year_pattern).sum())
})

proportion = proportion[proportion['% yop'] != 0]
proportion.sort_values('% yop', ascending=False)

,% yop,count yop
yop,66.281974,178846
ref,18.620521,42568
name,15.232820,30607
price,13.973079,37703
Unnamed: 0,11.941399,32221
model,1.183585,2846
size,0.187915,466


Analysis: 
* Name and Model can be used to extract yop
* Ref, Price, and Unnamed seem too irrelevant

In [13]:
def extract_year (row): 
    # Convert current yop to int
    yop = str(row['yop'])
    match = re.search (year_pattern, yop) 
    if match: 
        return int (match.group(1))

    # Extract yop from name 
    name = str(row['name'])
    match = re.search (year_pattern, name)
    if match: 
        return int (match.group(1))
    
    # Extract yop from model 
    model = str(row['model'])
    match = re.search (year_pattern, model) 
    if match: 
        return int (match.group(1))
    
    return np.nan

watches_cleaned['yop'] = watches_cleaned.apply (
    lambda row: extract_year(row), axis = 1
)

print ('check nan', watches_cleaned['yop'].isna().any())
print ('check null', watches_cleaned['yop'].isnull().any())
print ('Nan proportion: ', watches_cleaned['yop'].isna().mean()* 100)
print ('Nan count: ', watches_cleaned['yop'].isna().sum())

check nan True
check null True
Nan proportion:  30.661240947870112
Nan count:  82732


There are still 30.6% of yop is nan. 
# 
We can experiment if watches with the same model have same year of production. 

In [14]:
model_year_stats = watches_cleaned.dropna(subset=['yop']).groupby('model')['yop'].agg([
    'nunique',
    'count',
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
]).rename(columns= {'<lambda_0>': 'yop'})

model_year_stats = model_year_stats[model_year_stats['count'] >= 3]
model_year_stats['% unique'] = model_year_stats['nunique'] / model_year_stats['count']
print (model_year_stats.sort_values('% unique', ascending=True)); print()
print (model_year_stats['% unique'].describe())

                             nunique  count     yop  % unique
model                                                        
Datejust 41                       14   4948  2023.0  0.002829
Oyster Perpetual 41                4    703  2022.0  0.005690
Day-Date 40                       12   1477  2022.0  0.008125
Sky-Dweller                       16   1701  2022.0  0.009406
Black Bay Fifty-Eight              9    818  2021.0  0.011002
...                              ...    ...     ...       ...
Transocean 38                      4      4  1631.0  1.000000
TT3                                4      4  2008.0  1.000000
Artelier Skeleton                  5      5  2008.0  1.000000
Tradition                          3      3  2015.0  1.000000
Master Eight Days Perpetual        5      5  1612.0  1.000000

[874 rows x 4 columns]

count    874.000000
mean       0.365629
std        0.273625
min        0.002829
25%        0.123381
50%        0.323683
75%        0.555556
max        1.000000
Name: % u

Analysis: 
* In average, for watches with more than 4 ones having the sam  model, 25% of them have unique year of production. This means 75% of them have similar year of production

Conclusion: 
* We can use model as a feature to group the watches and find the year of production
* We should also do the same for 'reference' to see if model with the same reference have the same year of production

In [15]:
model_year_stats = watches_cleaned.dropna(subset=['yop']).groupby('ref')['yop'].agg([
    'nunique',
    'count',
    lambda x: x.mode()[0] if not x.mode().empty else np.nan, 
    lambda x: x.std() if not x.mode().empty else np.nan, 
]).rename(columns= {'<lambda_0>': 'yop', '<lambda_1>' : 'yop_std'})

model_year_stats = model_year_stats[model_year_stats['count'] >= 3]
model_year_stats['% unique'] = model_year_stats['nunique'] / model_year_stats['count']
print (model_year_stats.sort_values('% unique', ascending=True)); print()

print ('yop standard deviation: ') 
print (model_year_stats['yop_std'].describe()); print()

print ('% Unique: ')
print (model_year_stats['% unique'].describe())

                    nunique  count     yop   yop_std  % unique
ref                                                           
124300                    4    654  2022.0  0.867608  0.006116
126334                   11   1737  2023.0  1.668864  0.006333
126720VTNR                2    288  2023.0  0.452068  0.006944
126300                   11   1290  2022.0  1.594784  0.008527
126234                    7    782  2022.0  1.147281  0.008951
...                     ...    ...     ...       ...       ...
war201c.fc6266            3      3  2014.0  3.000000  1.000000
w7100055                  3      3  2015.0  2.516611  1.000000
w7100054                  3      3  2014.0  2.081666  1.000000
Y2431012/BE10/256S        3      3  2017.0  3.000000  1.000000
Y2431012.BE10.152A        4      4  2016.0  2.500000  1.000000

[11559 rows x 5 columns]

yop standard deviation: 
count    11559.000000
mean        11.590197
std         34.766513
min          0.000000
25%          0.500000
50%          1.5000

Analysis:
* In average, around 48% of watches with more than 3 of them having the same reference share different year of production
* For watches with different yop, they have 10-year difference in average, but the deviation can go up to 30 years

Conclusion: 
* The year of prod grouped with 'ref' is too risky, thereby not worth extracting

In [16]:
watches_cleaned['yop'] = watches_cleaned['yop'].fillna (
    watches_cleaned.groupby('model')['yop'].transform('median')
)

print ('Nan Porportion',  watches_cleaned['yop'].isna().mean() * 100)
print ('Nan Count', watches_cleaned['yop'].isna().sum())

Nan Porportion 4.471770696671188
Nan Count 12066


Drop the watches with nan values in 'yop'

In [17]:
watches_cleaned = watches_cleaned.dropna(subset=['yop'])

# Clean 'model' feature

In [18]:
print (watches['model'].sample(5)); print()
print (watches['name'].head(5)); print()

272104                       NaN
106106    Seamaster Planet Ocean
84924              HydroConquest
89907                  Avigation
39900              Santos Galbée
Name: model, dtype: str

0    Audemars Piguet Royal Oak Offshore Chronograph...
1    Audemars Piguet Royal Oak Selfwinding\n39mm Bl...
2    Audemars Piguet Royal Oak Chronograph\nBlue Di...
3    Audemars Piguet Royal Oak Chronograph\nSelfwin...
4    Audemars Piguet Royal Oak Offshore Chronograph...
Name: name, dtype: str



Analysis: 
* The model is often shown in the name of the watch

Conclusion: 
* We can use 'name' to group watches and impute 'model' 

In [19]:
model_list = watches_cleaned['model'].dropna().unique().tolist()
model_list = [m for m in model_list if m != 'Unknown']

def model_from_name (row): 
    if pd.isnull (row['model']): 
        name_str = str(row['name']).lower()
        for model in model_list: 
            if model.lower() in name_str: 
                return model 
    return row['model']

watches_cleaned['model'] = watches_cleaned.apply (model_from_name, axis = 1)
print ('Nan count', watches_cleaned['model'].isna().sum())
print ('% count', watches_cleaned['model'].isna().mean() * 100) 

Nan count 6793
% count 2.635397268777157


Drop the 2% of watches without model

In [20]:
watches_cleaned = watches_cleaned.dropna (subset = ['model'])

# Clean 'mvmt' feature 

Strategy: 
* Impute with 'ref' first - watches with the same reference must have the same movement
* Extract the movement from 'name'
* Impute 'name' with 'model' - we do this last because same model may have different types of movement 

In [21]:
watches['mvmt'].unique()

<StringArray>
[nan, 'Automatic', 'Manual winding', 'Quartz']
Length: 4, dtype: str

In [ ]:
watches_cleaned['mvmt'] = watches_cleaned['mvmt'].fillna (
    watches_cleaned.groupby('ref')['mvmt'].transform (
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)

def mvmt_from_name (row): 
    if pd.isnull (row['mvmt']): 
        name_lower = str(row['name']).lower()
        if any (word in name_lower for word in ['quartz', 'battery', 'solar', 'solor-powered', 'hybrid', 'spring-drive']): 
            return 'Quartz'
        elif any (word in name_lower for word in ['automatic',  'mechanical']): 
            return 'Automatic'
        elif any (word in name_lower for word in ['manual-winding', 'manual winding', 'self-winding', 'self winding']): 
            return 'Manual winding'
    return row['mvmt']
watches_cleaned['mvmt'] = watches_cleaned.apply (mvmt_from_name, axis = 1) 

watches_cleaned['mvmt'] = watches_cleaned['mvmt'].fillna (
    watches_cleaned.groupby('model')['mvmt'].transform (
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)

In [23]:
print ('Nan Count', watches_cleaned['mvmt'].isna().sum())
print ('% Count', watches_cleaned['mvmt'].isna().mean()* 100) 

Nan Count 35403
% Count 14.106635533755435


There are still up to 14% of nan values in this feature. Because movement is an important thing to determine the price of a watch, we will keep it. 
#
Conclusion: fill the remaining nan values with "Unknown"

In [24]:
watches_cleaned['mvmt'] = watches_cleaned['mvmt'].fillna ('Unknown') 
watches_cleaned['mvmt'].isna().any()

np.False_

# Clean 'casem' and 'bracem'

In [25]:
print (watches_cleaned['casem'].unique())
print()
print (watches_cleaned['bracem'].unique())

<StringArray>
[          nan,       'Steel',    'Titanium',   'Rose gold',    'Platinum',
  'White gold',     'Ceramic',      'Carbon', 'Yellow gold',  'Gold/Steel',
    'Red gold',    'Tantalum',    'Aluminum',      'Bronze',     'Plastic',
      'Silver',   'Palladium',    'Tungsten']
Length: 18, dtype: str

<StringArray>
[             nan,          'Steel',         'Rubber', 'Crocodile skin',
        'Leather',      'Rose gold',       'Platinum',     'Gold/Steel',
     'White gold',        'Ceramic',       'Titanium',        'Textile',
    'Yellow gold',      'Calf skin',       'Red gold',        'Plastic',
    'Lizard skin',      'Aluminium',        'Silicon',     'Shark skin',
         'Silver',     'Snake skin',          'Satin',   'Ostrich skin']
Length: 24, dtype: str


Analysis: 
* There are many different materials 
* Processing all these types of materials would dilute the data

Conclusion: 
* It is better to group them into levels of rarity

Strategy: 
* Split the price point into 3 ranges
* Calculate the average price for each material
* Sort the material into 3 categories according to the average price

In [26]:
# 65th percentile: common, 25th percentile: valuable, 10th percentile: rare. 
p65 = watches_cleaned['price'].quantile (0.65) 
p90 = watches_cleaned['price'].quantile (0.90) 

casem_avg_price = watches_cleaned.groupby('casem')['price'].mean() 
bracem_avg_price = watches_cleaned.groupby('bracem')['price'].mean() 

def categorize_material (avg_price): 
    if avg_price < p65: 
        return 'Common' 
    elif avg_price < p90: 
        return 'Valuable' 
    else: 
        return 'Rare' 
casem_map = {material : categorize_material(price) for material, price in casem_avg_price.items()}
bracem_map = {material : categorize_material(price) for material, price in bracem_avg_price.items()}

print ('Casem_map: ', casem_map)
print ('bracem_map: ', bracem_map)

Casem_map:  {'Aluminum': 'Valuable', 'Bronze': 'Common', 'Carbon': 'Rare', 'Ceramic': 'Valuable', 'Gold/Steel': 'Valuable', 'Palladium': 'Valuable', 'Plastic': 'Common', 'Platinum': 'Rare', 'Red gold': 'Valuable', 'Rose gold': 'Rare', 'Silver': 'Common', 'Steel': 'Common', 'Tantalum': 'Valuable', 'Titanium': 'Valuable', 'Tungsten': 'Common', 'White gold': 'Rare', 'Yellow gold': 'Valuable'}
bracem_map:  {'Aluminium': 'Common', 'Calf skin': 'Common', 'Ceramic': 'Rare', 'Crocodile skin': 'Valuable', 'Gold/Steel': 'Valuable', 'Leather': 'Valuable', 'Lizard skin': 'Valuable', 'Ostrich skin': 'Valuable', 'Plastic': 'Common', 'Platinum': 'Rare', 'Red gold': 'Valuable', 'Rose gold': 'Rare', 'Rubber': 'Valuable', 'Satin': 'Valuable', 'Shark skin': 'Valuable', 'Silicon': 'Common', 'Silver': 'Valuable', 'Snake skin': 'Valuable', 'Steel': 'Valuable', 'Textile': 'Common', 'Titanium': 'Valuable', 'White gold': 'Rare', 'Yellow gold': 'Rare'}


In [27]:
watches_cleaned ['casem'] = watches_cleaned['casem'].str.strip()
watches_cleaned['bracem'] = watches_cleaned['bracem'].str.strip()

watches_cleaned['casem'] = watches_cleaned['casem'].map (casem_map)
watches_cleaned['bracem']= watches_cleaned['bracem']. map (bracem_map)

In [28]:
def material_from_name (row, col_name): 
    if pd.isnull(row[col_name]) or str(row[col_name]).strip() == '': 
        name_lower = str(row['name']).lower()
        if col_name == 'casem':
            material_map = casem_map
        elif col_name == 'bracem': 
            material_map = bracem_map
        for keyword, material in material_map.items(): 
            if keyword.lower() in name_lower: 
                return material
    return row[col_name] 

watches_cleaned['casem'] = watches_cleaned.apply (
    lambda row: material_from_name(row, 'casem'), 
    axis = 1, 
)
watches_cleaned['bracem'] = watches_cleaned.apply (
    lambda row: material_from_name(row, 'bracem'), 
    axis = 1, 
)

def nan_report (df: pd.DataFrame):
    print (df.name)
    print ('Nan Count: ', df.isna().sum()) 
    print ('Nan proportion: ', df.isna().mean() * 100) 
    print()

nan_report(watches_cleaned['bracem']) 
nan_report(watches_cleaned['casem']) 

bracem
Nan Count:  131118
Nan proportion:  52.245115891730784

casem
Nan Count:  122626
Nan proportion:  48.86140408898381



Analysis: 
* 52% and 48% of data in bracem and casem is nan, respectively

# 

Strategy: 
* Impute the data by grouping the dataset with reference 


In [29]:
for col in ['casem', 'bracem']: 
    # Impute with reference
    watches_cleaned[col] = watches_cleaned[col].fillna (
        watches_cleaned.groupby('ref')[col].transform (
            lambda x: x.mode()[0] if not x.mode().empty else np.nan
        )
    )

nan_report (watches_cleaned['casem'])
nan_report (watches_cleaned['bracem'])

casem
Nan Count:  30816
Nan proportion:  12.278905194706875

bracem
Nan Count:  33755
Nan proportion:  13.449975494786168



Fill the remaining ~13% of casem and bracem with Unknown

In [30]:
for col in ['bracem', 'casem']: 
    watches_cleaned[col] = watches_cleaned[col].fillna ('Unknown')

# Clean 'sex' feature

Analysis: 
* The watches for female will usually have cheaper price
#
Strategy: 
* Map the sex feature to is_female feature 
* Impute using ref, mode, and size
* We can extract words like 'lady' from name to see if it's a female watch 

In [31]:
print (watches['sex'].sample (10).unique())

<StringArray>
[nan, 'Men's watch/Unisex', 'Women's watch']
Length: 3, dtype: str


In [32]:
# Map 'sex' to 'is_female' 
watches_cleaned ['is_female'] = watches_cleaned['sex'].map ({
    "Men's watch/Unisex": 0, 
    "Women's watch": 1 
})
female_keywords = ['lady', 'ladies', 'women', 'womens', 'woman', 'mother of pearl', 'mop']

#Extract female from name 
def female_from_name(name): 
    if pd.isna (name): return np.nan
    name_lower = str(name).lower() 
    return 1 if any (word in name_lower for word in female_keywords) else 0 

watches_cleaned['is_female'] = watches_cleaned['is_female'].fillna (
    watches_cleaned['name'].apply (female_from_name)
)

# Impute female from 'ref'
watches_cleaned['is_female'] = watches_cleaned['is_female'].fillna (
    watches_cleaned.groupby('ref')['is_female'].transform ( 
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)

# Impute female from 'model'
watches_cleaned['is_female'] = watches_cleaned['is_female'].fillna (
    watches_cleaned.groupby('model')['is_female'].transform ( 
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)

# Impute female from 'size'
watches_cleaned['is_female'] = watches_cleaned['is_female'].fillna (
    watches_cleaned.groupby('size')['is_female'].transform ( 
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)

nan_report(watches_cleaned['is_female'])

is_female
Nan Count:  955
Nan proportion:  0.38052811724250596



Analysis: 
* The nan proportion of 'is_female' is just ~3.8%
Conclusion: 
* It's worth exploring the data where 'is_female' is empty, and decide whether to fill with 'unknown' or drop

In [33]:
female_empty_df = watches_cleaned[watches_cleaned['is_female'].isna()] 
female_empty_df.sample (5)

,Unnamed: 0,name,price,brand,model,ref,mvmt,casem,bracem,yop,cond,sex,size,condition,is_female
154804,3295,NaN,65452.0,Rolex,Sea-Dweller,1665,Unknown,Common,Valuable,2020.0,NaN,NaN,NaN,Good,NaN
195517,44008,NaN,80000.0,Rolex,Day-Date 40,228206,Unknown,Rare,Rare,2022.0,NaN,NaN,NaN,Unworn,NaN
195288,43779,NaN,17300.0,Rolex,Submariner Date,126610LV,Unknown,Common,Valuable,2022.0,NaN,NaN,NaN,Unworn,NaN
186747,35238,NaN,11227.0,Rolex,Yacht-Master 40,16623,Unknown,Common,Valuable,2020.0,NaN,NaN,NaN,Very good,NaN
158122,6613,NaN,65454.0,Rolex,Submariner Date,ROLEX SUBMARINER 16808 NIPPLE,Unknown,Unknown,Unknown,1987.0,NaN,NaN,NaN,Very good,NaN


Analysis: 
* Most of the watches missing 'sex' are also missing many other features
#
Conclusion: 
* Drop the these watches is more reasonable

In [34]:
# Drop nan values from 'is_female' and drop the 'sex' feature 
watches_cleaned = watches_cleaned.dropna (subset = ['is_female'])
if 'sex' in watches_cleaned.columns: 
    watches_cleaned.drop ('sex', axis = 1, inplace = True) 

# 

# Clean 'cond' and 'condition'

In [35]:
condition_list = watches_cleaned['condition'].unique().tolist()
print (condition_list)

cond_list = watches_cleaned['cond'].unique().tolist()
print (cond_list)

[nan, 'Very good', 'Unworn', 'Good', 'Fair', 'New', 'Poor', 'Incomplete']
['Unworn', 'Very good', 'New', 'Good', 'Fair', nan, 'Poor', 'Incomplete']


* These two features are giving the same information
* It's better to group them into one feature

In [36]:
nan_report(watches_cleaned['cond'])
nan_report(watches_cleaned['condition'])

cond
Nan Count:  69255
Nan proportion:  27.700670367822344

condition
Nan Count:  183873
Nan proportion:  73.54566980784922



'cond' should be kept, and filled using 'condition', because it has less nan

In [37]:
watches_cleaned['cond'] = watches_cleaned['cond'].fillna (watches_cleaned['condition']) 
nan_report(watches_cleaned['cond'])

cond
Nan Count:  3116
Nan proportion:  1.2463401756715677



Analysis:
* 1.24% of cond is 'nan'
# 
Strategy:
* Watches with the same model and same price point should have the same condition
* Impute 'cond' with 'price' and 'model'

In [38]:
watches_cleaned['price_bins'] = pd.qcut (
    watches_cleaned['price'], 
    q = [0, 0.65, 0.90, 1.0], 
    labels = ['Common', 'Valuable', 'Rare']
)
watches_cleaned['cond'] = watches_cleaned['cond'].fillna (
    watches_cleaned.groupby(['price_bins', 'ref'])['cond'].transform (
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)
watches_cleaned['cond'] = watches_cleaned['cond'].fillna (
    watches_cleaned.groupby(['price_bins', 'model'])['cond'].transform (
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
)

nan_report(watches_cleaned['cond']) 

cond
Nan Count:  4
Nan proportion:  0.0015999232036862231



In [39]:
watches_cleaned.dropna (subset=['cond'], inplace = True)

In [40]:
cond_list = watches_cleaned['cond'].unique().tolist()
print (cond_list) 

#Analyze Conditions
cond_price_table = (
    watches_cleaned.dropna (subset=['cond', 'price']).groupby('cond').agg(
        count = ('price', 'size'), 
        median_price = ('price', 'median'), 
        mean_price = ('price', 'mean') 
    ).sort_values('median_price', ascending=False)
)
print (cond_price_table) 

['Unworn', 'Very good', 'New', 'Good', 'Fair', 'Poor', 'Incomplete']
            count  median_price    mean_price
cond                                         
Unworn      41823       12188.0  28690.139110
Very good   99783        8052.0  19065.829580
New         71015        6313.0  18592.793241
Good        30570        4347.0  10721.266961
Poor          139        2531.0   6761.258993
Fair         6645        2500.0   6224.487434
Incomplete     33         860.0   5642.000000


Map cond into different tags

In [ ]:
cond_map = {
    'Unworn': 'new', 
    'Very good': 'good', 
    'New': 'good', 
    'Good': 'good', 
    'Poor': 'bad', 
    'Fair': 'bad', 
    'Incomplete': 'bad', 
}

watches_cleaned['cond'] = watches_cleaned['cond'].map (cond_map)

# Clean 'size' feature 

In [59]:
watches['size'].sample (10) 

76633                 NaN
249082    40.1 x 48.15 mm
250124            38.5 mm
25416               40 mm
142451              33 mm
34306               29 mm
33622          29 x 41 mm
152810              41 mm
29653               46 mm
63463               39 mm
Name: size, dtype: str

The sisze usually has 2 format: 
* digit x digit mm
* digit mm
#
The floating point either uses '.' or ',' to seperate decimal numbers

In [42]:
watches_cleaned['size'] = watches_cleaned['size'].astype ('string').str.lower()
watches_cleaned['case_size_mm'] = watches_cleaned['size'].str.extract (r"(\d+(?:\.\d+)?)").astype ('float') 
size_max = watches_cleaned['case_size_mm'].max()
size_min = watches_cleaned['case_size_mm'].min()
print ('Size max:', size_max)
print ('Size min:', size_min); print()

Size max: 2724317425414.0
Size min: 0.0



Investigate watches with too small/big size

In [ ]:
oversize = watches_cleaned[(watches_cleaned['case_size_mm'] > 60)]
oversize['size']

913                                                   65 mm
914                                                   90 mm
1982                                                 334 mm
2128      reference number: 6022bc.oo.d002cr.01\n\n\n\n\...
2171      brand: audemars piguet\n\n\n\n\nmodel: "limite...
                                ...                        
281736                                               220 mm
282020    zenith defy 28800 automatic swiss made in 1970...
282035                                               100 mm
282189                                         400 x 135 mm
282537                                          120 x 90 mm
Name: size, Length: 748, dtype: string

The problem is caused by wrong formatting

In [ ]:
def extract_case_size (size): 
    if pd.isna (size): 
        return np.nan 
    
    text = str(size).strip().lower().replace (',', '.').replace ("´", '')

    #handle format digit x digit mm
    match = re.search (r'(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*mm', text)
    if match: 
        value = float (match.group (1))
        return value if 3 <= value <= 55 else np.nan
    
    #handle format: digit mm
    match = re.search (r'(\d+(?:\.\d+)?)\s*mm', text)
    if match: 
        value = float (match.group (1))
        return value if 3 <= value <= 55 else np.nan 
    
    return np.nan

watches_cleaned['case_size_mm'] = watches_cleaned['size'].apply (extract_case_size)
size_max = watches_cleaned['case_size_mm'].dropna().max()
size_min = watches_cleaned['case_size_mm'].dropna().min()
print ('Size max:' , size_max)
print ('Size min:' , size_min)

Size max: 55.0
Size min: 3.0


In [173]:
nan_report(watches_cleaned['case_size_mm'])

case_size_mm
Nan Count:  18702
Nan proportion:  7.480560622060094



Watches from the same model often has the same size 

In [179]:
watches_cleaned['case_size_mm'] = watches_cleaned['case_size_mm'].fillna ( 
    watches_cleaned.groupby('model')['case_size_mm'].transform ('median')
)
nan_report(watches_cleaned['case_size_mm'])

case_size_mm
Nan Count:  5
Nan proportion:  0.001999936002047934



In [183]:
watches_cleaned = watches_cleaned.dropna (subset = ['case_size_mm']) 

In [184]:
for col in ['price_bins', 'Unnamed: 0', 'condition', 'size']:
    if col in watches_cleaned.columns: 
        watches_cleaned = watches_cleaned.drop (col, axis = 1)
watches_cleaned.info()

<class 'pandas.DataFrame'>
Index: 250003 entries, 0 to 284490
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   name          182982 non-null  str    
 1   price         250003 non-null  float64
 2   brand         250003 non-null  str    
 3   model         250003 non-null  str    
 4   ref           220267 non-null  str    
 5   mvmt          250003 non-null  str    
 6   casem         250003 non-null  str    
 7   bracem        250003 non-null  str    
 8   yop           250003 non-null  float64
 9   cond          250003 non-null  str    
 10  is_female     250003 non-null  float64
 11  case_size_mm  250003 non-null  float64
dtypes: float64(4), str(8)
memory usage: 24.8 MB
